# Acoustic Scene Awareness — Full Experiment Suite (Google Colab)

**STW7088CEM — Artificial Neural Networks**  
Sudip Adhikari (250578)

Runs the complete 10-fold experiment suite on a free Colab **T4 GPU**.

Source: https://github.com/SudipAdh/acoustic-scene-awareness

---

### Before running

Enable the GPU: **Runtime → Change runtime type → Hardware accelerator → T4 GPU**

### What this notebook does

| Step | Time (T4) |
|---|---|
| Clone repo + install dependencies | ~2 min |
| Download UrbanSound8K + ESC-50 | ~5-15 min |
| Cache mel-spectrograms | ~2 min |
| Stage 1: dataset figures | <1 min |
| Stage 2: MLP vs CNN vs CNN+augment (10 folds each) | ~25 min |
| Stage 3: full evaluation + demo model | ~12 min |
| Stage 4: clustering (k-means + t-SNE) | ~3 min |
| Stage 5: autoencoder novelty detection | ~5 min |
| Package results for download | <1 min |

Total: roughly **50-65 minutes**.

## 1. Verify the GPU

If this reports no GPU, enable it via Runtime → Change runtime type before continuing.

In [ ]:
!nvidia-smi

import torch
print(f"\ntorch          : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU")

## 2. Clone the repository

In [ ]:
import os, shutil

REPO = "https://github.com/SudipAdh/acoustic-scene-awareness.git"
ROOT = "/content/acoustic-scene-awareness"

if os.path.exists(ROOT):
    shutil.rmtree(ROOT)
!git clone -q $REPO $ROOT

os.chdir(ROOT)
print("working directory:", os.getcwd())
!ls

## 3. Install dependencies

Colab already ships a CUDA-enabled PyTorch, so **torch is deliberately not reinstalled** —
installing the pinned CPU/MPS build from `requirements.txt` would silently remove GPU support.
Only the audio and analysis packages are installed.

In [ ]:
!pip install -q librosa soundfile

import librosa, sklearn, numpy, matplotlib, torch
print(f"librosa      : {librosa.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"numpy        : {numpy.__version__}")
print(f"torch        : {torch.__version__}  (CUDA: {torch.cuda.is_available()})")

## 4. Confirm the device the code will select

`src/config.py` picks MPS → CUDA → CPU. On Colab there is no MPS, so it should resolve to **cuda**.

In [ ]:
import sys
sys.path.insert(0, "/content/acoustic-scene-awareness")

from src import config as C
print("selected device :", C.DEVICE)
print("spectrogram     :", (C.N_MELS, C.N_FRAMES))
print("batch size      :", C.BATCH_SIZE)
print("epochs          :", C.EPOCHS)
print("folds           :", C.N_FOLDS)
assert C.DEVICE.type == "cuda", "Expected CUDA — check the GPU runtime setting"

## 5. Get the data

Two routes:

- **A (default): download in Colab.** Colab's network is far faster than a home connection, so the 5.6 GB UrbanSound8K archive usually arrives in a few minutes.
- **B (fallback): load a pre-computed spectrogram cache from Google Drive.** Only 383 MB, and skips both download and feature extraction. Use this if the download is slow or the runtime restarts.

Run **either** 5A or 5B, not both.

### 5A. Download the datasets (default)

In [ ]:
%%time
!bash scripts/download_data.sh

import subprocess
n = subprocess.run(
    "find data/raw/UrbanSound8K/audio -name '*.wav' | wc -l",
    shell=True, capture_output=True, text=True).stdout.strip()
print(f"\nUrbanSound8K wav files: {n} (expected 8732)")
assert int(n) > 8000, "Extraction incomplete — re-run this cell (the download resumes)."

### 5B. *Alternative:* load a cached feature archive from Google Drive

Skip this if 5A succeeded. To use it, upload `processed_features.tar.gz`
(produced by `scripts/package_features.sh` on your own machine) to the root of your Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# !mkdir -p data/processed
# !tar -xzf /content/drive/MyDrive/processed_features.tar.gz -C .
# !ls -la data/processed/

## 6. Cache the mel-spectrograms

Converts every clip to a 64×173 log-mel spectrogram once and stores it as `.npy`,
so the training stages read features straight from memory.
Skipped automatically if route 5B was used.

In [ ]:
%%time
!python run_experiments.py --stage prep

## 7. Stage 1 — Dataset exploration

Class balance, one example spectrogram per class, and the augmentation illustration.

In [ ]:
!python run_experiments.py --stage explore

from IPython.display import Image, display
for f in ["fig_class_distribution", "fig_spectrogram_examples", "fig_augmentation"]:
    display(Image(f"results/figures/{f}.png", width=950))

## 8. Stage 2 — Architecture comparison

MLP baseline vs CNN vs CNN with augmentation, each over the official 10 folds.
This is the longest stage.

In [ ]:
%%time
!python run_experiments.py --stage compare

In [ ]:
import json
from IPython.display import Image, display

res = json.load(open("results/logs/comparison_summary.json"))
print(f"{'model':<16}{'params':>12}{'accuracy':>16}{'macro F1':>18}")
print("-" * 62)
for name, r in res.items():
    print(f"{name:<16}{r['n_params']:>12,}"
          f"{r['acc_mean']*100:>11.2f} ±{r['acc_std']*100:<4.2f}"
          f"{r['f1_mean']:>13.4f} ±{r['f1_std']:<5.4f}")

display(Image("results/figures/fig_model_comparison.png", width=1000))
display(Image("results/figures/fig_training_curves.png", width=1000))

## 9. Stage 3 — Full evaluation

Pools held-out predictions across all ten folds — one honest prediction per clip —
then builds the aggregate confusion matrix and per-class scores. Also trains and
saves the model used by the live demo.

In [ ]:
%%time
!python run_experiments.py --stage final

In [ ]:
print(open("results/logs/final_classification_report.txt").read())

from IPython.display import Image, display
display(Image("results/figures/fig_confusion_matrix.png", width=800))
display(Image("results/figures/fig_per_class_f1.png", width=900))

## 10. Stage 4 — Clustering the learned representation

k-means and t-SNE over the CNN's 256-d embeddings, taken from held-out data only.
Tests whether the network organised the sound classes without using labels.

In [ ]:
%%time
!python run_experiments.py --stage cluster

In [ ]:
import json
from IPython.display import Image, display

cl = json.load(open("results/logs/cluster_metrics.json"))
m = cl["metrics"]
print(f"Adjusted Rand Index          : {m['adjusted_rand_index']:.4f}")
print(f"Normalised Mutual Information: {m['normalized_mutual_info']:.4f}")
print(f"Silhouette coefficient       : {m['silhouette']:.4f}\n")

print(f"{'cluster':<9}{'dominant class':<20}{'purity':>8}{'size':>7}")
print("-" * 46)
for cid, info in sorted(cl["cluster_purity"].items(), key=lambda kv: -kv[1]["purity"]):
    print(f"{cid:<9}{info['dominant_class']:<20}{info['purity']:>8.3f}{info['size']:>7}")

display(Image("results/figures/fig_tsne_embeddings.png", width=850))

## 11. Stage 5 — Novel-sound detection

Trains the convolutional autoencoder on known classes only, then measures how well
reconstruction error separates them from eight **unseen** ESC-50 household sounds.
Also evaluates the softmax-confidence baseline for comparison.

In [ ]:
%%time
!python run_experiments.py --stage anomaly

In [ ]:
import json
from IPython.display import Image, display

a = json.load(open("results/logs/anomaly_results.json"))
ae, base = a["autoencoder"], a["softmax_baseline"]

print("Novel-sound detection")
print(f"  autoencoder ROC-AUC        : {ae['roc_auc']:.4f}")
print(f"  autoencoder avg precision  : {ae['average_precision']:.4f}")
print(f"  precision / recall / F1    : {ae['precision']:.3f} / {ae['recall']:.3f} / {ae['f1']:.3f}")
print(f"\n  softmax baseline ROC-AUC   : {base['roc_auc']:.4f}")
print(f"  mean confidence on known   : {base['known_mean_conf']:.3f}")
print(f"  mean confidence on NOVEL   : {base['novel_mean_conf']:.3f}  <- confidently wrong")

for f in ["fig_anomaly_scores", "fig_anomaly_roc", "fig_ae_reconstructions"]:
    display(Image(f"results/figures/{f}.png", width=900))

## 12. Collect the results

Bundles every figure, metric file and trained model into a single archive and
downloads it. Unpack it over the local repository to fill in the report.

In [ ]:
!tar -czf /content/results_bundle.tar.gz results/figures results/logs models
!ls -lh /content/results_bundle.tar.gz

print("\nfigures:")
!ls results/figures/*.png | sed 's|^|  |'
print("metrics:")
!ls results/logs/*.json | sed 's|^|  |'

from google.colab import files
files.download("/content/results_bundle.tar.gz")

---

## Done

Unpack the downloaded bundle over the local checkout:

```bash
tar -xzf ~/Downloads/results_bundle.tar.gz -C path/to/acoustic-scene-awareness
```

The trained CNN (`models/cnn_demo.pt`) and autoencoder (`models/autoencoder.pt`)
are included, so the live microphone demo can be run locally:

```bash
.venv/bin/python demo/live_demo.py
```